In [1]:
!wget -O /kaggle/working/NITI_AI_Strategy.pdf https://www.niti.gov.in/sites/default/files/2023-03/National-Strategy-for-Artificial-Intelligence.pdf
!ls -lh /kaggle/working/*.pdf


--2026-10-12 18:52:34--  https://www.niti.gov.in/sites/default/files/2023-03/National-Strategy-for-Artificial-Intelligence.pdf
Resolving www.niti.gov.in (www.niti.gov.in)... 23.195.81.35, 23.195.81.41, 2600:1405:6400::17c3:513a, ...
Connecting to www.niti.gov.in (www.niti.gov.in)|23.195.81.35|:443... connected.
HTTP request sent, awaiting response... 403 Forbidden
2026-10-12 18:52:35 ERROR 403: Forbidden.

-rw-r--r-- 1 root root 0 Oct 12 18:52 /kaggle/working/NITI_AI_Strategy.pdf


In [2]:
!curl -L -A "Mozilla/5.0 (Windows NT 10.0; Win64; x64)" -o /kaggle/working/NITI_AI_Strategy.pdf https://www.niti.gov.in/sites/default/files/2023-03/National-Strategy-for-Artificial-Intelligence.pdf
!ls -lh /kaggle/working/*.pdf


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 4507k  100 4507k    0     0  1734k      0  0:00:02  0:00:02 --:--:-- 1734k
-rw-r--r-- 1 root root 4.5M Oct 12 18:54 /kaggle/working/NITI_AI_Strategy.pdf


In [4]:
!ls -lh /kaggle/input/*/*.pdf
!find /kaggle/input -name "*.pdf"


ls: cannot access '/kaggle/input/*/*.pdf': No such file or directory


In [5]:
!find /kaggle/input -name "*.pdf" -exec ls -lh {} \;


-rw-r--r-- 1 nobody nogroup 4.5M Oct 12 19:00 /kaggle/input/datasets/pailatejeswararao/niti-ai-strategy/NITI_AI_Strategy.pdf


In [6]:
!pip install -q pypdf
from pypdf import PdfReader
pdf_path = "/kaggle/input/datasets/pailatejeswararao/niti-ai-strategy/NITI_AI_Strategy.pdf"
r = PdfReader(pdf_path)
print(f"pages: {len(r.pages)}")
print(r.pages[1].extract_text()[:500])


pages: 114
 
National Strategy for Artificial Intelligence 
 
2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
This page has been intentionally left blank 
 
 
 
 
 
 
 
  


In [7]:
!rm /kaggle/working/NITI_AI_Strategy.pdf
!ls -lh /kaggle/input/datasets/pailatejeswararao/niti-ai-strategy/


total 4.5M
-rw-r--r-- 1 nobody nogroup 4.5M Oct 12 19:00 NITI_AI_Strategy.pdf


In [9]:
!pip install -q sentence-transformers faiss-cpu google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 18.6 MB/s eta 0:00:0000:0100:01


In [13]:
from kaggle_secrets import UserSecretsClient
import google.generativeai as genai
genai.configure(api_key=UserSecretsClient().get_secret("GEMINI_API_KEY"))

# list what YOU have access to
for m in genai.list_models():
    if 'flash' in m.name.lower():
        print(m.name)


models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [15]:
print(genai.GenerativeModel("models/gemini-3.8-flash").generate_content("say ok").text)


ok


In [16]:
# load the pdf file
from pypdf import PdfReader

pdf_path = "/kaggle/input/datasets/pailatejeswararao/niti-ai-strategy/NITI_AI_Strategy.pdf"
reader = PdfReader(pdf_path)

# read text page by page
pages = []
for i in range(len(reader.pages)):
    text = reader.pages[i].extract_text()
    if text is None:
        text = ""
    text = text.strip()
    if len(text) > 0:
        pages.append((i, text))

print("total pages with text:", len(pages))

# cut text into chunks
# each chunk is 1000 letters with 150 overlap
chunks = []
for page_num, page_text in pages:
    start = 0
    while start < len(page_text):
        end = start + 1000
        part = page_text[start:end].strip()
        # skip very small parts
        if len(part) > 200:
            chunk_id = "p" + str(page_num) + "_" + str(len(chunks))
            chunks.append({"id": chunk_id, "page": page_num, "text": part})
        start = start + 1000 - 150

print("total chunks:", len(chunks))
# show one chunk to check
print("sample id:", chunks[5]["id"])
print("sample page:", chunks[5]["page"])
print(chunks[5]["text"][:400])


total pages with text: 109
total chunks: 362
sample id: p4_5
sample page: 4
ackle some of the global challenges 
from AI’s perspective, be it application, research, development,  technology, or responsible AI. #AIforAll 
will focus on harnessing collaborations and partners hips, and aspires to ensure prosperity for all. Thus, 
#AIforAll means technology leadership in AI for achieving the greater good.  
While evolving the national strategy for AI, the underlying thrust wa


In [17]:
# load embedding model
# this model changes text into numbers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# make list of all chunk texts
texts = []
for c in chunks:
    texts.append(c["text"])

# convert texts into vectors
# show_progress_bar True will show time left
vectors = model.encode(texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True)

print("vectors shape:", vectors.shape)
# shape should be 362 by 384


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

vectors shape: (362, 384)


In [18]:
# build search index using faiss
# we use cosine similarity with normalized vectors
import faiss
import numpy as np

# make vectors unit length for cosine
lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
norm_vectors = vectors / lengths

# make index
size = norm_vectors.shape[1]
index = faiss.IndexFlatIP(size)
index.add(norm_vectors.astype("float32"))

print("total stored in index:", index.ntotal)


total stored in index: 362


In [19]:
# function to search top 10 chunks
import numpy as np

def search_top10(query):
    # change query into numbers using same model
    q = model.encode([query], convert_to_numpy=True)
    # make it unit length for cosine
    q = q / np.linalg.norm(q, axis=1, keepdims=True)
    # search in index
    scores, ids = index.search(q.astype("float32"), 10)
    # collect results
    results = []
    for rank in range(10):
        idx = int(ids[0][rank])
        results.append({
            "rank": rank + 1,
            "id": chunks[idx]["id"],
            "page": chunks[idx]["page"],
            "score": float(scores[0][rank]),
            "text": chunks[idx]["text"]
        })
    return results

# test with simple query
test_q = "What is AI for All strategy focus areas"
hits = search_top10(test_q)

# print ranks
for h in hits:
    print("rank", h["rank"], "id", h["id"], "page", h["page"], "score", round(h["score"], 4))
    print(h["text"][:250])
    print("")


rank 1 id p8_20 page 8 score 0.5857
National Strategy for Artificial Intelligence 
 
9 
 
partnerships? Bringing these questions into the open is the most important step in ensuring that AI 
advances create a better society. 
There has been tremendous activity concerning AI policy in d

rank 2 id p21_57 page 21 score 0.576
ing 
process, and add AI”  or “AI + X”; where “X”  can range from tasks such as driving a car, where AI can 
provide incremental value through improved routing and energy management, to act of sowing seeds, 
where AI can help inform decision making a

rank 3 id p11_23 page 11 score 0.5741
the physic al world . These human 
capabilities are augmented by the ability to learn from experience and keep adapting over time. AI 
systems are finding ever -wider application to supplement these capabilities across enterprises as they 
grow in so

rank 4 id p47_140 page 47 score 0.5721
National Strategy for Artificial Intelligence 
 
48 
 
 
 Way Forward to Harness the Power of 

In [20]:
# setup gemini model
# we use same model for Answer 1 and Answer 2
from kaggle_secrets import UserSecretsClient
import google.generativeai as genai

my_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
genai.configure(api_key=my_key)
llm = genai.GenerativeModel("models/gemini-3.8-flash")

# fixed prompt for both pipelines
# only chunks will change, prompt stays same
def make_prompt(query, top3):
    context = ""
    for i in range(3):
        context = context + "\nChunk " + str(i+1) + ":\n" + top3[i]["text"] + "\n"
    prompt = "You are a helper for NITI AI strategy document.\n"
    prompt = prompt + "Answer only from given context.\n"
    prompt = prompt + "If answer is not in context, say not found.\n"
    prompt = prompt + "\nContext:\n" + context + "\nQuestion: " + query + "\nAnswer:"
    return prompt

def get_answer1(query):
    hits = search_top10(query)
    top3 = hits[0:3]
    prompt = make_prompt(query, top3)
    out = llm.generate_content(prompt)
    return top3, out.text

# test
q = "What is AI for All strategy focus areas"
top3, ans1 = get_answer1(q)
print(ans1)
print("")
print("top3 ids used:")
for h in top3:
    print(h["rank"], h["id"], "page", h["page"])


not found

top3 ids used:
1 p8_20 page 8
2 p21_57 page 21
3 p11_23 page 11


In [21]:
# load cross encoder model
# this model reads query and chunk together
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# test with one pair
test_score = reranker.predict([["What is AI for All strategy focus areas", "Healthcare and agriculture are focus areas"]])
print(test_score)
print("model loaded ok")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[-3.2484143]
model loaded ok


In [22]:
# function to re rank top 10 using cross encoder
def rerank_top10(query, hits):
    # make pairs of query and chunk text
    pairs = []
    for h in hits:
        pairs.append([query, h["text"]])
    # get scores from cross encoder
    scores = reranker.predict(pairs)
    # add new score to each hit
    for i in range(len(hits)):
        hits[i]["cross_score"] = float(scores[i])
    # sort by new score, high to low
    ranked = sorted(hits, key=lambda x: x["cross_score"], reverse=True)
    return ranked

def get_answer2(query):
    hits = search_top10(query)
    ranked = rerank_top10(query, hits)
    top3_new = ranked[0:3]
    prompt = make_prompt(query, top3_new)
    out = llm.generate_content(prompt)
    return hits, ranked, top3_new, out.text

# three viva queries
# these cover trap, exception, and constraint
queries = [
    "What funding is proposed for Centres of Research Excellence COREs and ICTAIs",
    "What privacy safeguards and anonymization are needed for sharing health data in AI",
    "What barriers prevent small farmers with small land holdings from using AI in agriculture"
]

for q in queries:
    print("Question:", q)
    old_hits, new_ranked, new_top3, ans2 = get_answer2(q)
    # show old rank and new rank for top new chunks
    for new_pos in range(3):
        c = new_ranked[new_pos]
        # find old rank
        old_rank = 0
        for h in old_hits:
            if h["id"] == c["id"]:
                old_rank = h["rank"]
        print("chunk", c["id"], "page", c["page"], "old rank", old_rank, "new rank", new_pos+1)
    # also get answer 1 for compare
    top3_old, ans1 = get_answer1(q)
    print("Answer 1:")
    print(ans1[:600])
    print("Answer 2:")
    print(ans2[:600])
    print("")
    print("----")
    print("")


Question: What funding is proposed for Centres of Research Excellence COREs and ICTAIs
chunk p54_167 page 54 old rank 7 new rank 1
chunk p53_165 page 53 old rank 3 new rank 2
chunk p55_172 page 55 old rank 10 new rank 3
Answer 1:
Based on the provided context:

* **For ICTAIs:** The proposed initial funding is seed funding in the range of **INR 200 crore – INR 500 crore per ICTAI** for the first 5 years to cover major operational expenses, physical infrastructure, and technology/computing infrastructure. This seed funding is proposed to be a combination of funding from the government and private participation (under a PPP model), preferably through an equity/grant sharing mechanism.
* **For COREs (Centres of Research Excellence):** The proposed funding is **not found** in the context.
Answer 2:
Based on the provided context:

* **For COREs:** The proposed financial component could be to the tune of **INR 50 crore – INR 100 crore per CORE**, which should also include large-scale funding

In [23]:
# try simpler queries
simple_queries = [
    "COREs funding amount per centre",
    "data anonymization and privacy for AI",
    "challenges for farmers in adopting AI"
]

for q in simple_queries:
    print("Question:", q)
    old_hits, new_ranked, new_top3, ans2 = get_answer2(q)
    for new_pos in range(3):
        c = new_ranked[new_pos]
        old_rank = 0
        for h in old_hits:
            if h["id"] == c["id"]:
                old_rank = h["rank"]
        print("chunk", c["id"], "page", c["page"], "old", old_rank, "new", new_pos+1)
    top3_old, ans1 = get_answer1(q)
    print("Answer 1:")
    print(ans1[:700])
    print("Answer 2:")
    print(ans2[:700])
    print("")
    print("----")
    print("")


Question: COREs funding amount per centre
chunk p55_172 page 55 old 4 new 1
chunk p53_165 page 53 old 8 new 2
chunk p56_176 page 56 old 2 new 3
Answer 1:
Not found.
Answer 2:
Based on the context, the financial component for COREs could be to the tune of **INR 50 crore – INR 100 crore per CORE**.

----

Question: data anonymization and privacy for AI
chunk p87_287 page 87 old 2 new 1
chunk p86_282 page 86 old 4 new 2
chunk p86_286 page 86 old 7 new 3
Answer 1:
Based on the provided context, the key points regarding data anonymization and privacy for AI are:

* **Privacy Concerns:** Privacy concerns in AI largely stem from the collection and inappropriate use of data for personal discrimination. Traditional cryptographic methods fail to ensure that a user is not uniquely identifiable when personalized services are provided using collected or public data.
* **Research in Mathematical Models:** New mathematical models for privacy-preserving AI are being researched to reduce the risks of d

In [24]:
# check time for baseline and re rank
import time

q = "COREs funding amount per centre"

# time baseline search only
t1 = time.time()
for i in range(5):
    search_top10(q)
t2 = time.time()
base_time = (t2 - t1) / 5

# time re rank part only
hits = search_top10(q)
t1 = time.time()
for i in range(5):
    rerank_top10(q, hits)
t2 = time.time()
rerank_time = (t2 - t1) / 5

print("baseline search time in seconds:", round(base_time, 4))
print("rerank extra time in seconds:", round(rerank_time, 4))
print("total pipeline 2 time:", round(base_time + rerank_time, 4))


baseline search time in seconds: 0.0081
rerank extra time in seconds: 0.0382
total pipeline 2 time: 0.0463


In [25]:
# show text of top chunks for farmers query
q = "challenges for farmers in adopting AI"
hits = search_top10(q)
ranked = rerank_top10(q, hits)

for i in range(3):
    c = ranked[i]
    print("new rank", i+1, "id", c["id"], "page", c["page"])
    print(c["text"])
    print("")
    print("----")
    print("")


new rank 1 id p45_137 page 45
National Strategy for Artificial Intelligence 
 
46 
 
 
Key challenges to adoption of AI in India 
 Commonality of problems mandate an integrated approach  
 
The preceding analysis of focus sectors – Healthcare, Agriculture, Education, Smart Cities and 
Infrastructure, and Smart Mobility and Transport, highlight the potentia l of AI tools and technologies in 
transforming the sectors and state of Indian economy as a whole. The analysis, however, also detail a 
multitude of challenges that India needs to overcome to realise the full potential of a disruptive technology 
like AI. 
Adopting a narrow view and focusing on the challenges for a specific sector, the barriers to developing a 
robust set of AI applications may seem contextual and limited to that sector. Taking Healthcare sector as 
an example, enabling large scale adoption would require at least the following factors to be addressed:  
a) absence of collaborative effort between various stakeholder

In [26]:
# fixed third query uses exact words from chunk
q = "What barriers need to be addressed to achieve the goals of AI for All"

old_hits, new_ranked, new_top3, ans2 = get_answer2(q)

for new_pos in range(3):
    c = new_ranked[new_pos]
    old_rank = 0
    for h in old_hits:
        if h["id"] == c["id"]:
            old_rank = h["rank"]
    print("chunk", c["id"], "page", c["page"], "old", old_rank, "new", new_pos+1)

top3_old, ans1 = get_answer1(q)
print("Answer 1:")
print(ans1)
print("")
print("Answer 2:")
print(ans2)


chunk p6_11 page 6 old 1 new 1
chunk p45_137 page 45 old 4 new 2
chunk p72_233 page 72 old 7 new 3


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash
Please retry in 3h40m36.11286337s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.8-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 13236
}
]

In [29]:
# test new key with correct name
from kaggle_secrets import UserSecretsClient
import google.generativeai as genai

my_key2 = UserSecretsClient().get_secret("newkey3")
genai.configure(api_key=my_key2)
llm = genai.GenerativeModel("models/gemini-3.8-flash")
print(llm.generate_content("say ok").text)


ok


In [30]:
# use lite model for tonight
# final report will use 3.8 again after reset
llm = llm2

# fixed third query
q = "What barriers need to be addressed to achieve the goals of AI for All"

old_hits, new_ranked, new_top3, ans2 = get_answer2(q)

for new_pos in range(3):
    c = new_ranked[new_pos]
    old_rank = 0
    for h in old_hits:
        if h["id"] == c["id"]:
            old_rank = h["rank"]
    print("chunk", c["id"], "page", c["page"], "old", old_rank, "new", new_pos+1)

top3_old, ans1 = get_answer1(q)
print("Answer 1:")
print(ans1)
print("")
print("Answer 2:")
print(ans2)


NameError: name 'llm2' is not defined

In [31]:
# check what is still in memory
try:
    print(len(chunks))
except:
    print("chunks gone, need rerun")

try:
    print(llm2)
except:
    print("llm2 gone, need remake")


362
llm2 gone, need remake


In [32]:
# remake lite model only
from kaggle_secrets import UserSecretsClient
import google.generativeai as genai

my_key3 = UserSecretsClient().get_secret("newkey3")
genai.configure(api_key=my_key3)
llm2 = genai.GenerativeModel("models/gemini-flash-lite-latest")
llm = llm2
print(llm.generate_content("say ok").text)


ok


In [33]:
# fixed third query with lite model
q = "What barriers need to be addressed to achieve the goals of AI for All"

old_hits, new_ranked, new_top3, ans2 = get_answer2(q)

for new_pos in range(3):
    c = new_ranked[new_pos]
    old_rank = 0
    for h in old_hits:
        if h["id"] == c["id"]:
            old_rank = h["rank"]
    print("chunk", c["id"], "page", c["page"], "old", old_rank, "new", new_pos+1)

top3_old, ans1 = get_answer1(q)
print("Answer 1:")
print(ans1)
print("")
print("Answer 2:")
print(ans2)


chunk p6_11 page 6 old 1 new 1
chunk p45_137 page 45 old 4 new 2
chunk p72_233 page 72 old 7 new 3
Answer 1:
Based on the provided context, the barriers that need to be addressed to achieve the goals of #AIforAll are:

* Lack of broad-based expertise in research and application of AI
* Absence of enabling data ecosystems – access to intelligent data
* High resource cost and low awareness for adoption of AI
* Privacy and security, including a lack of formal regulations around anonymisation of data
* Inadequate availability of AI expertise, manpower and skilling opportunities
* High resource cost and low awareness for adopting AI in business processes
* Unclear privacy, security and ethical regulations
* Unattractive Intellectual Property regime to incentivise research and adoption of AI

Answer 2:
Based on the provided context, the barriers that need to be addressed to achieve the goals of #AIforAll are:

a) Lack of broad based expertise in research and application of AI, 
b) Absence of

In [34]:
# latency with LLM time
import time

q = "COREs funding amount per centre"

t1 = time.time()
top3_old, ans1 = get_answer1(q)
t2 = time.time()
time_p1 = t2 - t1

t1 = time.time()
old_hits, new_ranked, new_top3, ans2 = get_answer2(q)
t2 = time.time()
time_p2 = t2 - t1

print("Pipeline 1 total with LLM:", round(time_p1, 3))
print("Pipeline 2 total with LLM:", round(time_p2, 3))


Pipeline 1 total with LLM: 0.377
Pipeline 2 total with LLM: 0.467


In [35]:
# two new queries for step 3
new_queries = [
    "What restriction applies when sharing sensitive health data for AI development",
    "Why can AI adoption not succeed without data ecosystems and skilled workforce"
]

for q in new_queries:
    print("Question:", q)
    hits = search_top10(q)
    ranked = rerank_top10(q, hits)
    for new_pos in range(10):
        c = ranked[new_pos]
        old_rank = 0
        for h in hits:
            if h["id"] == c["id"]:
                old_rank = h["rank"]
        if new_pos < 3:
            print("chunk", c["id"], "page", c["page"], "old", old_rank, "new", new_pos+1)
    print("")


Question: What restriction applies when sharing sensitive health data for AI development
chunk p45_138 page 45 old 1 new 1
chunk p87_287 page 87 old 5 new 2
chunk p88_291 page 88 old 6 new 3

Question: Why can AI adoption not succeed without data ecosystems and skilled workforce
chunk p7_15 page 7 old 8 new 1
chunk p71_231 page 71 old 3 new 2
chunk p109_347 page 109 old 5 new 3



In [36]:
# answers for the two new queries with lite model
llm = llm2

new_queries = [
    "What restriction applies when sharing sensitive health data for AI development",
    "Why can AI adoption not succeed without data ecosystems and skilled workforce"
]

for q in new_queries:
    print("Question:", q)
    top3_old, ans1 = get_answer1(q)
    old_hits, new_ranked, new_top3, ans2 = get_answer2(q)
    print("Answer 1:")
    print(ans1[:800])
    print("")
    print("Answer 2:")
    print(ans2[:800])
    print("")
    print("----")
    print("")


Question: What restriction applies when sharing sensitive health data for AI development
Answer 1:
Based on the provided context, the challenges regarding data usage include data usage without consent, the risk of identification of individuals through data, data selection bias, and asymmetry in data aggregation. To address ethical concerns regarding data sharing, it becomes necessary for firms (such as diagnostic centres acting as data custodians) to take the consent of individuals before aggregating and selling their data, ensuring appropriate permissions are taken from the actual data owners. Additionally, the document mentions establishing data protection frameworks and sectoral regulatory frameworks, and promoting the adoption of international standards.

Answer 2:
Not found.

----

Question: Why can AI adoption not succeed without data ecosystems and skilled workforce
Answer 1:
Not found.

Answer 2:
Based on the provided context, access to data and a skilled workforce are addresse